# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error


In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
df=spaceship.dropna(axis=0)

In [4]:
mapping={k:k[0] for k in df['Cabin'].unique()}
df['Cabin']=df['Cabin'].map(mapping)
df=df.drop(['PassengerId','Name'],axis=1)

C:\Users\jesus\AppData\Local\Temp\ipykernel_32488\1321247650.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Cabin']=df['Cabin'].map(mapping)


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [5]:
df.select_dtypes('object').columns
non_num_col=['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']
ohe = OneHotEncoder(sparse_output=False)
ohe.fit(df[non_num_col])

OneHotEncoder(sparse_output=False)

**Perform Train Test Split**

In [6]:
features = df.drop(columns='Transported')
target = df["Transported"]
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

# Normalizing

In [7]:
ohe = OneHotEncoder(drop='first',sparse_output=False)
ohe.fit(X_train)

OneHotEncoder(drop='first', sparse_output=False)

In [8]:
object_col=X_train.select_dtypes('object').columns

In [9]:
X_train_trans_np = ohe.transform(X_train[object_col])
X_train_trans_df = pd.DataFrame(X_train_trans_np, columns=ohe.get_feature_names_out(), index=X_train.index)
X_train_final=pd.concat([X_train_trans_df,X_train],axis=1)
X_train=X_train.drop(object_col, axis=1) 


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Age
- FoodCourt
- RoomService
- ShoppingMall
- Spa
- ...


In [ ]:
X_test_trans_np = ohe.transform(X_test[object_col])
X_test_trans_df = pd.DataFrame(X_test_trans_np, columns=ohe.get_feature_names_out(), index=X_test.index)
X_test_final=pd.concat([X_test_trans_df,X_test],axis=1)
X_test=X_test.drop(object_col, axis=1) 

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [ ]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100, # number of models to use
                               max_samples = 1000)

In [ ]:
bagging_reg.fit(X_train, y_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=20),
                 max_samples=1000, n_estimators=100)

In [ ]:
y_pred_test_bag = bagging_reg.predict(X_test)

print(f"Results for Decision tree")
print(f"MAE {mean_absolute_error(y_pred_test_bag, y_test): .2f}")
print(f"MSE {mean_squared_error(y_pred_test_bag, y_test): .2f}")
print(f"RMSE {root_mean_squared_error(y_pred_test_bag, y_test): .2f}")
print(f"R2 score {bagging_reg.score(X_test, y_test): .2f}")

Results for Decision tree
MAE  0.31
MSE  0.16
RMSE  0.40
R2 score  0.36


- Random Forests

In [ ]:
forest = RandomForestRegressor(n_estimators=100,
                             max_depth=20)
forest.fit(X_train, y_train)

RandomForestRegressor(max_depth=20)

In [ ]:
y_pred_test_rf = forest.predict(X_test)

print(f"Results for Random Forest")
print(f"MAE, {mean_absolute_error(y_pred_test_rf, y_test): .2f}")
print(f"MSE, {mean_squared_error(y_pred_test_rf, y_test): .2f}")
print(f"RMSE, {root_mean_squared_error(y_pred_test_rf, y_test): .2f}")
print(f"R2 score, {forest.score(X_test, y_test): .2f}")

Results for Random Forest
MAE,  0.31
MSE,  0.17
RMSE,  0.41
R2 score,  0.34


- Adaptive Boosting

In [ ]:
ada_reg = AdaBoostRegressor(DecisionTreeRegressor(max_depth=20),
                            n_estimators=100)
ada_reg.fit(X_train, y_train)

AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=20),
                  n_estimators=100)

In [ ]:
y_pred_test_ada = ada_reg.predict(X_test)

print(f"Results for adaptive booster")
print(f"MAE, {mean_absolute_error(y_pred_test_ada, y_test): .2f}")
print(f"MSE, {mean_squared_error(y_pred_test_ada, y_test): .2f}")
print(f"RMSE, {root_mean_squared_error(y_pred_test_ada, y_test): .2f}")
print(f"R2 score, {ada_reg.score(X_test, y_test): .2f}")

Results for adaptive booster
MAE,  0.34
MSE,  0.23
RMSE,  0.48
R2 score,  0.08


- Gradient Boosting

In [ ]:
gb_reg = GradientBoostingRegressor(max_depth=20,
                                   n_estimators=100)
gb_reg.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=20)

In [ ]:
y_pred_test_gb = gb_reg.predict(X_test)

print(f"Results for Gradient booster")
print(f"MAE, {mean_absolute_error(y_pred_test_gb, y_test): .2f}")
print(f"MSE, {mean_squared_error(y_pred_test_gb, y_test): .2f}")
print(f"RMSE, {root_mean_squared_error(y_pred_test_gb, y_test): .2f}")
print(f"R2 score, {gb_reg.score(X_test, y_test): .2f}")

Results for Gradient booster
MAE,  0.31
MSE,  0.22
RMSE,  0.47
R2 score,  0.10


Which model is the best and why?

I would pick the Decision Tree based on the results. It is the one with the lowest MAE, MSE and RMSE. Also, it has the highest R2 score.